# Notebook 05 — Wikibase Upload

**Project:** Linked Open Exhibition — NFDI4Culture / Hochschule Hannover (BIM-126-02)  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6)  
**Depends on:**
- `02_dnb_filter_exhibitions.ipynb` → `sprengel_exhibitions.csv`
- `04_wikibase_data_model.ipynb` → `wikibase_property_map.json`
- `.env` file with Wikibase credentials

**Purpose:** Create one Wikibase item per exhibition record in the CSV. Each item receives statements for title, dates, location, GND ID, and DNB IDN. Idempotent: records that already exist (matched by DNB IDN) are skipped.

---

## How `wikibaseintegrator` works

`wikibaseintegrator` is a Python library that wraps the Wikibase MediaWiki API. The workflow for creating an item is:

1. Create a new item object: `wbi.item.new()`
2. Set labels and descriptions
3. Add statements using `datatypes` helpers (e.g. `datatypes.Item`, `datatypes.Time`, `datatypes.ExternalID`)
4. Call `.write()` to send the item to the server

Statements reference properties by their local P-ID (from `wikibase_property_map.json`).

In [1]:
import os, json, time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from wikibaseintegrator import WikibaseIntegrator, wbi_login
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator import datatypes

load_dotenv(Path("../../.env"))

WB_URL  = os.getenv("WB_URL", "https://wikibase.wbworkshop.tibwiki.io")
WB_USER = os.getenv("WB_USER")
WB_PASS = os.getenv("WB_PASSWORD")

if not WB_USER or not WB_PASS:
    raise EnvironmentError("Set WB_USER and WB_PASSWORD in your .env file.")

wbi_config["MEDIAWIKI_API_URL"]   = f"{WB_URL}/w/api.php"
wbi_config["SPARQL_ENDPOINT_URL"] = f"{WB_URL}/query/sparql"
wbi_config["WIKIBASE_URL"]        = WB_URL

login_instance = wbi_login.Login(user=WB_USER, password=WB_PASS)
wbi = WikibaseIntegrator(login=login_instance)
print(f"Connected to: {WB_URL}")

Connected to: https://wikibase.wbworkshop.tibwiki.io


## Step 1 — Load data

In [2]:
CSV_PATH  = Path("../sprengel_exhibitions.csv")
MAP_PATH  = Path("../wikibase_property_map.json")

df  = pd.read_csv(CSV_PATH)
wbmap = json.loads(MAP_PATH.read_text(encoding="utf-8"))
props = wbmap["properties"]
items = wbmap["items"]

P_INSTANCE_OF   = props["instance of"]
P_TITLE         = props["title"]
P_START_DATE    = props["start date"]
P_END_DATE      = props["end date"]
P_LOCATION      = props["location"]
P_GND_ID        = props.get("GND ID", "")
P_DNB_IDN       = props["DNB IDN"]

Q_EXHIBITION    = items["Exhibition"]
Q_SPRENGEL      = items["Sprengel Museum Hannover"]
Q_EXH_CAT       = items["Exhibition Catalogue"]

print(f"Loaded {len(df)} records")
print(f"Properties: {props}")

Loaded 582 records
Properties: {'instance of': 'P1', 'title': 'P2', 'start date': 'P3', 'end date': 'P4', 'location': 'P5', 'GND ID': 'P6', 'DNB IDN': 'P7', 'exhibition catalogue': 'P8'}


## Step 2 — Idempotency check helper

Before creating an item, search existing items by DNB IDN to avoid duplicates.

In [3]:
import requests as req

def find_item_by_idn(idn):
    """Return QID of an existing item with this DNB IDN, or None."""
    sparql = f"""
        SELECT ?item WHERE {{
          ?item wdt:{P_DNB_IDN} "{idn}" .
        }} LIMIT 1
    """
    try:
        resp = req.get(
            wbi_config["SPARQL_ENDPOINT_URL"],
            params={"query": sparql, "format": "json"},
            timeout=15,
        )
        bindings = resp.json()["results"]["bindings"]
        if bindings:
            return bindings[0]["item"]["value"].split("/")[-1]
    except Exception:
        pass
    return None

## Step 3 — Upload exhibitions

Each record in the CSV becomes one Wikibase item. The item is labelled with the catalogue title and tagged as an `Exhibition Catalogue` (the DNB records are catalogues, not the exhibitions themselves). The Sprengel Museum is added as location.

In [ ]:
created  = 0
skipped  = 0
errors   = 0

for _, row in df.iterrows():
    idn   = str(row.get("idn", "")).strip()
    title = str(row.get("title", "")).strip()
    year  = str(row.get("year", "")).strip()

    if not idn or not title:
        errors += 1
        continue

    # Idempotency: skip if already uploaded
    existing = find_item_by_idn(idn)
    if existing:
        print(f"  Skip {idn} — already exists as {existing}")
        skipped += 1
        continue

    try:
        statements = [
            datatypes.Item(prop_nr=P_INSTANCE_OF, value=Q_EXH_CAT),
            datatypes.Item(prop_nr=P_LOCATION,    value=Q_SPRENGEL),
            datatypes.ExternalID(prop_nr=P_DNB_IDN, value=idn),
        ]

        # Add year as start date if available
        if year and len(year) == 4 and year.isdigit():
            statements.append(
                datatypes.Time(prop_nr=P_START_DATE, time=f"+{year}-00-00T00:00:00Z", precision=9)
            )

        # Add ISBN / URL as GND ID placeholder if GND ID property exists
        gnd = str(row.get("isbn", "")).strip()
        if P_GND_ID and gnd:
            statements.append(datatypes.ExternalID(prop_nr=P_GND_ID, value=gnd))

        item = wbi.item.new()
        item.labels.set(language="en", value=title[:250])  # max label length
        item.descriptions.set(language="en", value=f"Exhibition catalogue, Sprengel Museum Hannover, {year}")
        item.claims.add(statements)
        item.write(summary="Bot: upload DNB exhibition catalogue record")

        print(f"  Created {item.id}: {title[:60]}")
        created += 1

    except Exception as e:
        print(f"  ERROR for IDN {idn}: {e}")
        errors += 1

    time.sleep(0.5)

print(f"\nCreated: {created} | Skipped: {skipped} | Errors: {errors}")

  Created Q6: Niki de Saint Phalle - Die Grotte
  Created Q7: Niki de Saint Phalle - The Grotto
  Created Q8: Feministische Avantgarde
  Created Q9: Grethe Jürgens
  Created Q10: Lillien Grupe - Realität(en)?
  Created Q11: Love you for infinity


Error while writing to the Wikibase instance
Traceback (most recent call last):
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\entities\baseentity.py", line 287, in _write
    json_result: dict = edit_entity(data=data, id=entity_id, type=self.type, summary=summary, clear=clear, is_bot=is_bot, allow_anonymous=allow_anonymous,
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\wbi_helpers.py", line 339, in edit_entity
    return mediawiki_api_call_helper(data=params, is_bot=is_bot, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\wbi_helpers.py", line 221, in mediawiki_api_call_helper
    return mediawiki_api_call('POST', mediawiki_api_url=mediawiki_api_url,

  ERROR for IDN 1375750437: 'Item [[Item:Q11|Q11]] already has label "Love you for infinity" associated with language code en, using the same description text.'
  Created Q13: On lies, secrets and silence
  Created Q14: Peter Heber - über das Sterben
  Created Q15: Verfemt - gehandelt
  Created Q16: Bastian Hoffmann - radical negation
  Created Q17: How to be an artist like me
  Created Q18: Nahaufnahmen
  Created Q19: Peter Tuma - aufkommende Unruhe
  Created Q20: Porträt einer Sammlung - Sprengel Museum Hannover
  Created Q21: Subjective evidence
  Created Q22: Adrian Sauer: truth table
  Created Q23: Christian Retschlag
  Created Q24: Fotografien der Fotografie
  Created Q25: Kunst und Künstler*innen in Hannover im Nationalsozialismus
  Created Q26: ocular witness
  Created Q27: Picasso, Beckmann
  Created Q28: Portrait of a collection - Sprengel Museum Hannover
  Created Q29: Schweinebewusstsein
  Created Q30: The real thing
  Created Q31: Welche Moderne?
  Created Q32: Edda Z

Connection error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Sleeping for 60 seconds.
Traceback (most recent call last):
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\urllib3\connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\urllib3\connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\http\client.py", line 1395, in getresponse
    response.begin()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x6

  Created Q35: Vom Beginnen
  Created Q36: A00121
  Created Q37: Danach ist davor
  Created Q38: Elementarteile


...

  ERROR for IDN 960878556: 'Item [[Item:Q278|Q278]] already has label "Sprengel zu 2000" associated with language code en, using the same description text.'


Error while writing to the Wikibase instance
Traceback (most recent call last):
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\entities\baseentity.py", line 287, in _write
    json_result: dict = edit_entity(data=data, id=entity_id, type=self.type, summary=summary, clear=clear, is_bot=is_bot, allow_anonymous=allow_anonymous,
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\wbi_helpers.py", line 339, in edit_entity
    return mediawiki_api_call_helper(data=params, is_bot=is_bot, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\wbi_helpers.py", line 221, in mediawiki_api_call_helper
    return mediawiki_api_call('POST', mediawiki_api_url=mediawiki_api_url,

  ERROR for IDN 96087853X: 'Item [[Item:Q278|Q278]] already has label "Sprengel zu 2000" associated with language code en, using the same description text.'


Error while writing to the Wikibase instance
Traceback (most recent call last):
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\entities\baseentity.py", line 287, in _write
    json_result: dict = edit_entity(data=data, id=entity_id, type=self.type, summary=summary, clear=clear, is_bot=is_bot, allow_anonymous=allow_anonymous,
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\wbi_helpers.py", line 339, in edit_entity
    return mediawiki_api_call_helper(data=params, is_bot=is_bot, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\git\linked-open-exhibition\.venv\Lib\site-packages\wikibaseintegrator\wbi_helpers.py", line 221, in mediawiki_api_call_helper
    return mediawiki_api_call('POST', mediawiki_api_url=mediawiki_api_url,

  ERROR for IDN 960878564: 'Item [[Item:Q278|Q278]] already has label "Sprengel zu 2000" associated with language code en, using the same description text.'


---

**Next step:** Run `06_mediawiki_upload.ipynb` to upload cover images to MediaWiki and link them to the Wikibase items.